# Módulo 9 · Tarea 3: Procesamiento y análisis

Limpieza, transformación y análisis de `data/tareas.csv`. Al final se guarda `data/tareas_limpias.csv`, que usa el notebook de visualización.

In [1]:
import pandas as pd

RUTA_CSV = "../data/tareas.csv"
RUTA_LIMPIO = "../data/tareas_limpias.csv"
PRIORIDADES = ["baja", "media", "alta"]
FECHA_CORTE = pd.Timestamp.now()

crudo = pd.read_csv(RUTA_CSV)
df = crudo.copy()
print("Registros originales:", len(df))
df.head()

Registros originales: 10


,id,titulo,estado,fecha_creacion,categoria,fecha_limite,prioridad,tiempo_estimado,completado_en
0,1,Estudiar SQLite,completada,2026-09-09 23:06:31,General,NaN,media,NaN,NaN
1,4,practicar formularios,pendiente,2026-09-29 18:51:15,Estudio,NaN,media,NaN,NaN
2,5,Lavar ropa,completada,2026-09-01 09:00:00,Hogar,2026-09-02 20:00:00,media,60.0,2026-09-01 10:20:00
3,6,Hacer el súper,completada,2026-09-03 10:00:00,compras,2026-09-04 20:00:00,Alta,90.0,2026-09-03 11:45:00
4,7,Pagar la luz,completada,2026-09-05 08:00:00,Personal,2026-09-08 18:00:00,alta,15.0,2026-09-10 09:00:00


## 1. Diagnóstico de calidad

In [2]:
print("Valores vacíos por columna:")
print(df.isna().sum().to_string())
print()
print("Filas duplicadas (sin contar el id):", df.drop(columns="id").duplicated().sum())
print("Prioridades distintas:", df["prioridad"].unique().tolist())
print("Categorías distintas:", df["categoria"].unique().tolist())
print("Tiempos estimados negativos:", (df["tiempo_estimado"] < 0).sum())

Valores vacíos por columna:
id                 0
titulo             0
estado             0
fecha_creacion     0
categoria          0
fecha_limite       2
prioridad          0
tiempo_estimado    3
completado_en      4

Filas duplicadas (sin contar el id): 0
Prioridades distintas: ['media', 'Alta', 'alta', 'urgente', 'baja', ' BAJA']
Categorías distintas: ['General', 'Estudio', 'Hogar', 'compras', 'Personal', 'Ejercicio', 'HOGAR ', 'Mascotas']
Tiempos estimados negativos: 1


## 2. Limpieza de datos incompletos o inconsistentes

1. Quitar espacios sobrantes en textos y unificar mayúsculas.
2. Prioridades fuera de `baja / media / alta` → `media` (el valor por defecto del modelo).
3. Fechas inválidas → `NaT` (fecha vacía).
4. Tiempos estimados negativos → vacío.
5. Eliminar registros duplicados.
6. Fechas de término anteriores a la creación → vacío, porque son imposibles.

In [3]:
df["titulo"] = df["titulo"].str.strip()
df["estado"] = df["estado"].str.strip().str.lower()
df["categoria"] = df["categoria"].str.strip().str.capitalize()

df["prioridad"] = df["prioridad"].str.strip().str.lower()
prioridad_invalida = ~df["prioridad"].isin(PRIORIDADES)
print("Prioridades inválidas corregidas:", prioridad_invalida.sum())
df.loc[prioridad_invalida, "prioridad"] = "media"

for columna in ["fecha_creacion", "fecha_limite", "completado_en"]:
    df[columna] = pd.to_datetime(df[columna], errors="coerce")
print("Fechas límite inválidas:", (df["fecha_limite"].isna() & crudo["fecha_limite"].notna()).sum())

negativos = df["tiempo_estimado"] < 0
print("Tiempos negativos eliminados:", negativos.sum())
df.loc[negativos, "tiempo_estimado"] = pd.NA

antes = len(df)
df = df.drop_duplicates(subset=df.columns.drop("id"), keep="first")
print("Duplicados eliminados:", antes - len(df))

termino_imposible = df["completado_en"] < df["fecha_creacion"]
print("Fechas de término imposibles:", termino_imposible.sum())
df.loc[termino_imposible, "completado_en"] = pd.NaT

Prioridades inválidas corregidas: 1
Fechas límite inválidas: 1
Tiempos negativos eliminados: 1
Duplicados eliminados: 0
Fechas de término imposibles: 1


In [4]:
completada_sin_fecha = (df["estado"] == "completada") & df["completado_en"].isna()
print("Tareas completadas sin fecha de término:", completada_sin_fecha.sum())
df[completada_sin_fecha][["id", "titulo", "estado", "completado_en"]]

Tareas completadas sin fecha de término: 2


,id,titulo,estado,completado_en
0,1,Estudiar SQLite,completada,NaT
7,10,Limpiar la cocina,completada,NaT


Estas tareas se conservan para los conteos, pero no se usan en los cálculos de duración porque no se sabe cuándo terminaron.

In [5]:
df["prioridad"] = pd.Categorical(df["prioridad"], categories=PRIORIDADES, ordered=True)
df["tiempo_estimado"] = df["tiempo_estimado"].astype("Float64")

print("Registros después de limpiar:", len(df))
df.dtypes

Registros después de limpiar: 10


id                          int64
titulo                        str
estado                        str
fecha_creacion     datetime64[us]
categoria                     str
fecha_limite       datetime64[us]
prioridad                category
tiempo_estimado           Float64
completado_en      datetime64[us]
dtype: object

## 3. Transformación: duración real

`duracion_real_min = completado_en − fecha_creacion`, expresada en minutos y en horas. Mide el tiempo desde que se creó la tarea hasta que se completó.

In [6]:
df["duracion_real_min"] = (df["completado_en"] - df["fecha_creacion"]).dt.total_seconds() / 60
df["duracion_real_h"] = df["duracion_real_min"] / 60
df["diferencia_min"] = df["duracion_real_min"] - df["tiempo_estimado"]

df[df["completado_en"].notna()][
    ["titulo", "tiempo_estimado", "duracion_real_min", "duracion_real_h", "diferencia_min"]
].head(10).round(1)

,titulo,tiempo_estimado,duracion_real_min,duracion_real_h,diferencia_min
2,Lavar ropa,60.0,80.0,1.3,20.0
3,Hacer el súper,90.0,105.0,1.8,15.0
4,Pagar la luz,15.0,7260.0,121.0,7245.0
5,Hacer la comida,45.0,60.0,1.0,15.0
6,Salir a correr,40.0,50.0,0.8,10.0


## 4. Tiempo estimado vs real por prioridad

Se muestra el promedio y la mediana; la mediana es menos sensible a tareas atípicas.

In [7]:
con_duracion = df[df["duracion_real_min"].notna() & df["tiempo_estimado"].notna()]

comparacion = con_duracion.groupby("prioridad", observed=False).agg(
    tareas=("id", "count"),
    estimado_promedio_min=("tiempo_estimado", "mean"),
    real_promedio_min=("duracion_real_min", "mean"),
    estimado_mediana_min=("tiempo_estimado", "median"),
    real_mediana_min=("duracion_real_min", "median"),
)
comparacion["real_vs_estimado"] = comparacion["real_mediana_min"] / comparacion["estimado_mediana_min"]
comparacion.round(1)

,tareas,estimado_promedio_min,real_promedio_min,estimado_mediana_min,real_mediana_min,real_vs_estimado
prioridad,,,,,,
baja,1,40.0,50.0,40.0,50.0,1.2
media,2,52.5,70.0,52.5,70.0,1.3
alta,2,52.5,3682.5,52.5,3682.5,70.1


## 5. Evaluación de retrasos y variable `is_delayed`

- **Completada:** hay retraso si `completado_en > fecha_limite`.
- **Pendiente:** hay retraso si la `fecha_limite` ya pasó.
- **Sin fecha límite:** no se puede evaluar y se marca como `False`.

In [8]:
completada = df["completado_en"].notna()

retraso_completada = completada & (df["completado_en"] > df["fecha_limite"])
retraso_pendiente = (df["estado"] == "pendiente") & (df["fecha_limite"] < FECHA_CORTE)

df["is_delayed"] = retraso_completada | retraso_pendiente
df["horas_de_retraso"] = ((df["completado_en"] - df["fecha_limite"]).dt.total_seconds() / 3600).where(retraso_completada)

print(f"Tareas con retraso: {df['is_delayed'].sum()} de {len(df)} ({df['is_delayed'].mean() * 100:.1f} %)")
print("Tareas sin fecha límite (no evaluables):", df["fecha_limite"].isna().sum())

Tareas con retraso: 2 de 10 (20.0 %)
Tareas sin fecha límite (no evaluables): 3


In [9]:
df.groupby("prioridad", observed=False).agg(
    tareas=("id", "count"),
    con_retraso=("is_delayed", "sum"),
    porcentaje_retraso=("is_delayed", "mean"),
    horas_retraso_promedio=("horas_de_retraso", "mean"),
).assign(porcentaje_retraso=lambda t: t["porcentaje_retraso"] * 100).round(1)

,tareas,con_retraso,porcentaje_retraso,horas_retraso_promedio
prioridad,,,,
baja,3,0,0.0,NaN
media,5,1,20.0,NaN
alta,2,1,50.0,39.0


In [10]:
df[df["is_delayed"]][["id", "titulo", "estado", "prioridad", "fecha_limite", "completado_en", "horas_de_retraso"]].round({"horas_de_retraso": 1})

,id,titulo,estado,prioridad,fecha_limite,completado_en,horas_de_retraso
4,7,Pagar la luz,completada,alta,2026-09-08 18:00:00,2026-09-10 09:00:00,39.0
8,11,Ir al banco,pendiente,media,2026-09-25 14:00:00,NaT,NaN


## 6. Tareas atípicas con estadísticas descriptivas

Se usa el rango intercuartílico (IQR): un valor es atípico si está por debajo de `Q1 − 1.5·IQR` o por encima de `Q3 + 1.5·IQR`.

In [11]:
df[["tiempo_estimado", "duracion_real_min"]].describe().round(1)

,tiempo_estimado,duracion_real_min
count,6.0,5.0
mean,46.7,1511.0
std,26.0,3213.9
min,15.0,50.0
25%,32.5,60.0
50%,42.5,80.0
75%,56.2,105.0
max,90.0,7260.0


In [12]:
def limites_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return max(q1 - 1.5 * iqr, 0), q3 + 1.5 * iqr

bajo_est, alto_est = limites_iqr(df["tiempo_estimado"].dropna())
bajo_real, alto_real = limites_iqr(df["duracion_real_min"].dropna())
print(f"Tiempo estimado normal: entre {bajo_est:.0f} y {alto_est:.0f} min")
print(f"Duración real normal:   entre {bajo_real:.0f} y {alto_real:.0f} min")

df["es_atipica"] = (
    ~df["tiempo_estimado"].between(bajo_est, alto_est) & df["tiempo_estimado"].notna()
) | (
    ~df["duracion_real_min"].between(bajo_real, alto_real) & df["duracion_real_min"].notna()
)

df[df["es_atipica"]][["id", "titulo", "tiempo_estimado", "duracion_real_min", "duracion_real_h"]].round(1)

Tiempo estimado normal: entre 0 y 92 min
Duración real normal:   entre 0 y 172 min


,id,titulo,tiempo_estimado,duracion_real_min,duracion_real_h
4,7,Pagar la luz,15.0,7260.0,121.0


## 7. Resultado final y guardado

Variables nuevas: `duracion_real_min`, `duracion_real_h`, `diferencia_min`, `is_delayed`, `horas_de_retraso` y `es_atipica`.

In [13]:
df.to_csv(RUTA_LIMPIO, index=False)
print(f"Dataset limpio guardado en {RUTA_LIMPIO}: {len(df)} filas, {df.shape[1]} columnas")
df.head()

Dataset limpio guardado en ../data/tareas_limpias.csv: 10 filas, 15 columnas


,id,titulo,estado,fecha_creacion,categoria,fecha_limite,prioridad,tiempo_estimado,completado_en,duracion_real_min,duracion_real_h,diferencia_min,is_delayed,horas_de_retraso,es_atipica
0,1,Estudiar SQLite,completada,2026-09-09 23:06:31,General,NaT,media,<NA>,NaT,NaN,NaN,<NA>,False,NaN,False
1,4,practicar formularios,pendiente,2026-09-29 18:51:15,Estudio,NaT,media,<NA>,NaT,NaN,NaN,<NA>,False,NaN,False
2,5,Lavar ropa,completada,2026-09-01 09:00:00,Hogar,2026-09-02 20:00:00,media,60.0,2026-09-01 10:20:00,80.0,1.333333,20.0,False,NaN,False
3,6,Hacer el súper,completada,2026-09-03 10:00:00,Compras,2026-09-04 20:00:00,alta,90.0,2026-09-03 11:45:00,105.0,1.750000,15.0,False,NaN,False
4,7,Pagar la luz,completada,2026-09-05 08:00:00,Personal,2026-09-08 18:00:00,alta,15.0,2026-09-10 09:00:00,7260.0,121.000000,7245.0,True,39.0,True
